<!-- CONCLUSION-CELL -->
> ## 🎯 결론 — 컨디션 분류: **AUC 0.65로 "좋은날/나쁜날" 구분 가능** (프로젝트의 실용 결과물)
>
> 회귀 R²(0.034)는 낮지만, whiff% **상위33%=좋음 / 하위33%=나쁨** 이진분류로 전환하니 쓸 만해짐.
>
> | | val AUC | test AUC |
> |---|---|---|
> | **정형 XGBClassifier** ⭐ | **0.685** | 0.645 |
> | 정형+영상 | 0.66 | 0.57 (무의, p=0.009) |
>
> - **"정확한 whiff% 값"은 못 맞혀도 "오늘 컨디션 좋은 날인가"는 구분 가능** → 프로젝트 정의("하던 만큼 나오나")와 일치.
> - **투수 유형별(E10)**: 파워형 0.63 > 핀포인트형 0.58. 구속이 무기인 투수일수록 판별 정확하나, **전 유형에서 랜덤(0.5)보다 나아 활용 가능** (회귀는 핀포인트형서 R²<0로 불가) → 분류 채택이 실용적으로도 옳았음.
> - **컨디션 신호(SHAP)**: 구속 · **구속 변동성(std)** · 회전수 · 평소 대비(prev).

# 19. 컨디션 분류 (whiff% 좋은날/나쁜날)

**전환 동기**: 18번에서 whiff% 회귀 R²=0.036(낮음)이지만 **분류 AUC=0.637**(쓸만). 
정확한 값 예측은 어려워도 "오늘 컨디션 좋은 날인가"는 구분 가능 → 프로젝트 정의("하던 만큼 나오나")와 일치.

**설계**
- 라벨: whiff% **train 분포 상위 33%=좋음(1) / 하위 33%=나쁨(0)**, 중간 제외 (극단 대비 명확)
- 누수 방지: 분위수 경계는 **train만**으로 산출
- 모델: XGBClassifier (정형 vs 정형+영상 비교)
- 평가: ROC-AUC, PR-AUC, 정밀도/재현율/F1, confusion, val+test
- 영상 융합: 30 seed AUC paired t-test (분류에서도 영상 효과 검증)
- SHAP: 무엇이 '좋은 날' 신호인가

## 1. 설정 + 병합 (18번과 동일 데이터)

In [ ]:
import os
import numpy as np
import pandas as pd

IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive; drive.mount('/content/drive')
    BASE='/content/drive/MyDrive/MLB_pitcher'; DATA=f'{BASE}/data'
    STAT_DIR=next((d for d in [f'{DATA}/features',f'{DATA}/4_features'] if os.path.exists(f'{d}/features_pitch15.parquet')), f'{DATA}/features')
    BIO_DIR=f'{DATA}/4_features'; TARGETS_PATH=f'{DATA}/game_targets.parquet'; OUTPUT_DIR=f'{BASE}/4_output'
else:
    BASE=r'c:\Users\suyou\OneDrive\Desktop\ASAC\PROJECT\투수 컨디션 예측'
    STAT_DIR=os.path.join(BASE,'0_data','4_features'); BIO_DIR=STAT_DIR
    TARGETS_PATH=os.path.join(BASE,'0_data','game_targets.parquet'); OUTPUT_DIR=os.path.join(BASE,'4_output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

STATCAST_PATH=os.path.join(STAT_DIR,'features_pitch15.parquet')
BIO_PATH=os.path.join(BIO_DIR,'features_bio_pitch15_full9.parquet')
KEYS=['game_pk','pitcher','season']
TARGET='y_whiff'           # 컨디션 대표 (18번 결과 AUC 최고)
TRAIN_SEASONS=[2021,2022,2023]; VAL_SEASON=2024; TEST_SEASON=2025
LOW_Q, HIGH_Q = 0.33, 0.67  # 하위33%=나쁨, 상위33%=좋음
N_SEEDS=30

df_stat=pd.read_parquet(STATCAST_PATH); df_bio=pd.read_parquet(BIO_PATH); tgt=pd.read_parquet(TARGETS_PATH)
for d in (df_stat,df_bio,tgt):
    for k in KEYS:
        if k in d.columns: d[k]=d[k].astype('int64')
DROP=set(KEYS)|{'y_whiff','y_xwoba','y_fip','y_era'}
STAT_COLS=[c for c in df_stat.columns if c not in DROP]
BIO_COLS=[c for c in df_bio.columns if c not in (set(KEYS)|{'n_pitches_used'})]
df=(df_stat[KEYS+STAT_COLS].merge(df_bio[KEYS+BIO_COLS],on=KEYS,how='inner')
    .merge(tgt[KEYS+[TARGET]],on=KEYS,how='inner')).dropna(subset=[TARGET]).reset_index(drop=True)
ALL_COLS=STAT_COLS+BIO_COLS
print(f'병합 {len(df):,}경기 | 정형 {len(STAT_COLS)} + 영상 {len(BIO_COLS)}')

## 2. 라벨 생성 (train 분위수 기준) + 분할

상위33%=좋음(1) / 하위33%=나쁨(0). 중간 33%는 모호하므로 제외.

In [ ]:
tr_mask=df['season'].isin(TRAIN_SEASONS)
lo,hi=df.loc[tr_mask,TARGET].quantile([LOW_Q,HIGH_Q])
print(f'라벨 경계(train): 나쁨≤{lo:.3f} / 좋음≥{hi:.3f}')

def label(s):
    y=pd.Series(np.nan,index=s.index)
    y[s>=hi]=1; y[s<=lo]=0
    return y
df['label']=label(df[TARGET])
d=df.dropna(subset=['label']).copy(); d['label']=d['label'].astype(int)

tr=d[d['season'].isin(TRAIN_SEASONS)]; vl=d[d['season']==VAL_SEASON]; te=d[d['season']==TEST_SEASON]
print(f'\n분류 표본: train {len(tr)} / val {len(vl)} / test {len(te)}')
for nm,part in [('train',tr),('val',vl),('test',te)]:
    print(f'  {nm} 좋음비율: {part["label"].mean():.2f}')

## 3. 분류 모델 학습 (정형 vs 융합) + 평가

In [ ]:
try:
    import xgboost as xgb, shap
except ImportError:
    import subprocess; subprocess.run(['pip','install','-q','xgboost','shap']); import xgboost as xgb, shap
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_score, recall_score, confusion_matrix, classification_report)

CLF_PARAMS=dict(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2, reg_lambda=1.0,
                eval_metric='auc', random_state=42, n_jobs=-1, verbosity=0)

def fit_eval(cols, label=''):
    m=xgb.XGBClassifier(**CLF_PARAMS)
    m.fit(tr[cols], tr['label'], eval_set=[(vl[cols], vl['label'])], verbose=False)
    pv=m.predict_proba(vl[cols])[:,1]; pt=m.predict_proba(te[cols])[:,1]
    out=dict(name=label, n_feat=len(cols),
             val_auc=roc_auc_score(vl['label'],pv), val_ap=average_precision_score(vl['label'],pv),
             test_auc=roc_auc_score(te['label'],pt), test_ap=average_precision_score(te['label'],pt))
    # 임계 0.5 기준 분류 리포트(val)
    pred=(pv>=0.5).astype(int)
    out.update(val_f1=f1_score(vl['label'],pred), val_prec=precision_score(vl['label'],pred),
               val_rec=recall_score(vl['label'],pred))
    return m, out, pv

m_stat, r_stat, pv_stat = fit_eval(STAT_COLS, '정형')
m_fuse, r_fuse, pv_fuse = fit_eval(ALL_COLS,  '정형+영상')
rep=pd.DataFrame([r_stat,r_fuse])
pd.set_option('display.float_format', lambda v:f'{v:.4f}')
print('=== 분류 성능 ===')
print(rep[['name','n_feat','val_auc','val_ap','val_f1','val_prec','val_rec','test_auc','test_ap']].to_string(index=False))
print('\n=== 정형 모델 confusion(val, 임계0.5) ===')
print(confusion_matrix(vl['label'], (pv_stat>=0.5).astype(int)))
print(classification_report(vl['label'], (pv_stat>=0.5).astype(int), target_names=['나쁨','좋음']))

## 4. 영상 융합 효과 — AUC paired t-test (30 seed)

In [ ]:
from scipy import stats
auc_s, auc_f = [], []
for seed in range(N_SEEDS):
    p={**CLF_PARAMS,'random_state':seed}
    ms=xgb.XGBClassifier(**p); ms.fit(tr[STAT_COLS],tr['label'],eval_set=[(vl[STAT_COLS],vl['label'])],verbose=False)
    auc_s.append(roc_auc_score(vl['label'], ms.predict_proba(vl[STAT_COLS])[:,1]))
    mf=xgb.XGBClassifier(**p); mf.fit(tr[ALL_COLS],tr['label'],eval_set=[(vl[ALL_COLS],vl['label'])],verbose=False)
    auc_f.append(roc_auc_score(vl['label'], mf.predict_proba(vl[ALL_COLS])[:,1]))
auc_s, auc_f = np.array(auc_s), np.array(auc_f)
t,pv=stats.ttest_rel(auc_f, auc_s)
print('=== 영상 융합 분류 AUC (paired t-test) ===')
print(f'정형      : AUC {auc_s.mean():.4f} ± {auc_s.std():.4f}')
print(f'정형+영상 : AUC {auc_f.mean():.4f} ± {auc_f.std():.4f}')
print(f'차이 {auc_f.mean()-auc_s.mean():+.4f}  t={t:.3f} p={pv:.4f}  '
      f'{"✅영상 유의" if (pv<0.05 and auc_f.mean()>auc_s.mean()) else "❌영상 무의/음수"}')

## 5. SHAP — 무엇이 '좋은 날' 신호인가 (정형 모델)

In [ ]:
expl=shap.TreeExplainer(m_stat)
sv=expl(vl[STAT_COLS])
imp=pd.Series(np.abs(sv.values).mean(axis=0), index=STAT_COLS).sort_values(ascending=False)
print('=== 컨디션(좋은날) 분류 — SHAP Top 15 ===')
print(imp.head(15).to_string())

import matplotlib.pyplot as plt
top=imp.head(15)[::-1]
plt.figure(figsize=(8,6))
plt.barh(range(len(top)), top.values, color='steelblue')
plt.yticks(range(len(top)), top.index, fontsize=9)
plt.xlabel('mean |SHAP|'); plt.title('whiff 컨디션 분류 — 핵심 신호 Top15')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'condition_clf_shap.png'),dpi=150,bbox_inches='tight'); plt.show()

## 6. 저장 + 결론

In [ ]:
import joblib, json
joblib.dump(m_stat, os.path.join(OUTPUT_DIR,'condition_clf_stat.pkl'))
rep.to_csv(os.path.join(OUTPUT_DIR,'condition_classification_results.csv'), index=False)
summary=dict(target=TARGET, label_boundary=dict(bad_le=float(lo), good_ge=float(hi)),
             n_train=len(tr), n_val=len(vl), n_test=len(te),
             stat_val_auc=float(r_stat['val_auc']), stat_test_auc=float(r_stat['test_auc']),
             fuse_val_auc=float(r_fuse['val_auc']),
             video_auc_diff=float(auc_f.mean()-auc_s.mean()), video_auc_p=float(pv),
             top_signals=imp.head(5).index.tolist())
json.dump(summary, open(os.path.join(OUTPUT_DIR,'condition_clf_summary.json'),'w'), ensure_ascii=False, indent=2)

print('=== 결론 ===')
print(f"정확한 whiff% 예측은 어렵지만(회귀 R²~0.036), 컨디션 좋은날/나쁜날 분류는")
print(f"  AUC = {r_stat['val_auc']:.3f}(val) / {r_stat['test_auc']:.3f}(test) 로 구분 가능.")
print(f"영상 융합: 분류에서도 {'유의 개선' if (pv<0.05 and auc_f.mean()>auc_s.mean()) else '효과 없음'} (p={pv:.3f}).")
print(f"핵심 신호: {imp.head(5).index.tolist()}")
print('\n저장: condition_clf_stat.pkl / condition_classification_results.csv / condition_clf_summary.json')

## 7. E10 투수 유형별 분류 AUC — "어떤 투수의 컨디션을 잘 구분하나" (2026-07-12)

회귀 오차분석(13번 E9)에서 파워형(R²+0.018) vs 핀포인트형(-0.024) 차이 발견 → **분류 AUC에서도 재현되나?**
AUC는 값예측이 아니라 순위구분이라 R² 착시가 없음. cell7의 `m_stat`(정형 분류모델)·`d`·`tr/te` 재사용.

> 목적: "이 모델을 어떤 투수에게 신뢰하고 쓸 수 있나"의 실전 적용 범위 확정.

In [ ]:
# E10. 투수 유형별 분류 AUC (cell7의 m_stat / d / tr / te / STAT_COLS 재사용)
def _auc_of(part, mask=None, name='', min_n=60):
    g = part if mask is None else part[mask]
    if len(g) < min_n or g['label'].nunique() < 2:
        return None
    p = m_stat.predict_proba(g[STAT_COLS])[:,1]
    return {'group':name,'n':len(g),'좋음비율':round(g['label'].mean(),2),
            'AUC':round(roc_auc_score(g['label'], p),4)}

print('='*58); print('[전체]'); print('='*58)
for nm, part in [('val 전체', vl), ('test 전체', te)]:
    r=_auc_of(part, name=nm); print(f'  {r}' if r else f'  {nm}: 표본부족')

print('\n' + '='*58)
print('[투수 유형별 test AUC] — 회귀 결과(E9) 재현되나?')
print('='*58)
rows=[]
for feat,hn,ln in [('avg_speed_Fastball','파워형(FB구속>med)','핀포인트형(FB구속<=med)'),
                   ('std_speed_all','구속변동큰','구속안정'),
                   ('avg_spin_Fastball','스핀많은','스핀적은')]:
    if feat in te.columns:
        md=tr[feat].median()   # train 기준 분할(leakage 방지)
        rows.append(_auc_of(te, te[feat]>md, hn))
        rows.append(_auc_of(te, te[feat]<=md, ln))
res=pd.DataFrame([r for r in rows if r])
print(res.to_string(index=False))
res.to_csv(os.path.join(OUTPUT_DIR,'clf_by_pitcher_type.csv'), index=False)
print('\n저장: clf_by_pitcher_type.csv')
print('[결론] 파워형 AUC 0.63 > 핀포인트형 0.58 → 회귀와 일관. 전 유형 AUC 0.58~0.65로 "완전 안됨" 그룹은 없음(핀포인트형이 상대적 약).')